# Algorithms 15: Reduced Row Echelon Form**Learning Objectives:**1. Implement the RREF algorithm iteratively using helper functions2. Compute the Inverse of a matrix much faster using augmented RREF3. Compute Determinants in $O(N^3)$ time instead of $O(N!)$**Prerequisites:** Algorithms 14**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 15

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "15_reduced_row_echelon_form",    "level": 3,    "title": "Algorithms 15: Reduced Row Echelon Form",    "prerequisites": [        "Algorithms 14"    ],    "skills_taught": [        "lesson.algorithms_15_reduced_row_echelon_form"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "16_k_means_clustering.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_15_reduced_row_echelon_form",        "title": "Lesson Algorithms 15 Reduced Row Echelon Form",        "notebook": "15_reduced_row_echelon_form.ipynb",        "level": 3    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "15_reduced_row_echelon_form.ipynb",        "level": 3    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 14.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_15_reduced_row_echelon_form', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport math, randomprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The RREF Algorithm> 📖 *From Algorithms Ch. 15:* To convert a matrix to RREF:1. Initialize `pivot_row = 0` and `pivot_col = 0`.2. Look for a non-zero entry in `pivot_col` at or below `pivot_row`.3. If found:    - Swap that row with `pivot_row`.   - Divide `pivot_row` by the pivot element so it becomes 1.   - Subtract multiples of `pivot_row` from all other rows so the rest of the column becomes 0.   - Increment both `pivot_row` and `pivot_col`.4. If NOT found: Increment only `pivot_col`.5. Repeat until you run out of rows or columns.### Inverse via RREFTo invert matrix $A$, augment it with the Identity matrix $I$: $[A | I]$. Run RREF. If the left side becomes $I$, the right side is $A^{-1}$.### Fast Determinant via RREFRecursive determinant is $O(N!)$. RREF takes $O(N^3)$ time.- Swapping two rows multiplies determinant by $-1$.- Dividing a row by $c$ multiplies determinant by $c$.If you track these changes during RREF, the final matrix has determinant $1$, and you can multiply backward to find the original determinant!

### Comprehension Check ✅1. If you augment $A = [2]$ with $I = [1]$ and run RREF, what do you get?2. During RREF, you swap rows twice, and divide one row by 3, and another by 4. If the resulting RREF matrix is the identity, what was the original determinant?<details><summary>💡 Explanation after a retrieval attempt</summary>1. $[2 | 1]$. Divide by 2: $[1 | 0.5]$. So $A^{-1} = [0.5]$.2. Determinant of identity is 1. Original determinant = $1 \times (-1)^2 \times 3 \times 4 = 12$.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: RREF by HandConvert $\begin{bmatrix} 2 & 4 \\ 3 & 1 \end{bmatrix}$ to RREF on paper.Track the operations you performed.*(Work it out on paper, then check mentally)*

In [ ]:
# Expected sequence of operations:# 1. Pivot at (0,0) is 2. Divide Row 0 by 2 -> [1, 2]# 2. Eliminate Row 1: Row 1 = Row 1 - 3*Row 0 -> [0, -5]# 3. Pivot at (1,1) is -5. Divide Row 1 by -5 -> [0, 1]# 4. Eliminate Row 0: Row 0 = Row 0 - 2*Row 1 -> [1, 0]# Result is the Identity matrix.

---## Phase 1 — Algorithm Construction### Step 1: Helper FunctionsWrite three helper methods for your `Matrix` class:1. `swap_rows(self, i, j)`2. `multiply_row(self, i, c)`3. `add_multiple_of_row(self, target_row, source_row, c)` (does: `target = target + c * source`)

In [ ]:
class Matrix:    def __init__(self, data):        self.data = [list(row) for row in data]        self.num_rows = len(self.data)        self.num_cols = len(self.data[0]) if self.num_rows > 0 else 0            # YOUR HELPER FUNCTIONS HERE    pass# Test your helpers!

### Step 2: The RREF AlgorithmImplement `rref(self)` using the exact procedure described in the Phase -1 section.

In [ ]:
# Add this to your Matrix class:# def rref(self):#     # Create a copy so we don't mutate the original#     rref_mat = Matrix(self.data)#     pr = 0#     pc = 0#     while pr < rref_mat.num_rows and pc < rref_mat.num_cols:#         # Find a non-zero element in column `pc` at or below row `pr`#         # YOUR CODE HERE#         pass#     return rref_mat# Test:# m = Matrix([[2, 4], [3, 1]])# print(m.rref().data) # Should be [[1.0, 0.0], [0.0, 1.0]]

### Step 3: Fast Determinant (Interleaved Practice)Copy-paste your RREF code into a new method `fast_determinant(self)`. Track a variable `det = 1`. Whenever you swap, `det *= -1`. Whenever you divide by $c$, `det *= c`.At the end, if the RREF matrix is the Identity, return `det`. Otherwise (e.g. it has a row of zeros), the determinant is 0.

In [ ]:
# def fast_determinant(self):#     # YOUR CODE HERE#     pass

---## Phase 2 — Experimental Verification### Pathological Case: Float ImprecisionWhen doing division, floats can become slightly inaccurate (e.g., `0.0000000000000001` instead of `0`). If you check `if val == 0:`, it might fail! You must check `if abs(val) < 1e-10:`.Go back and update your RREF implementation to use this tolerance threshold!

In [ ]:
# Test your float tolerance:# m = Matrix([[3, 6], [1, 2]])# The determinant is exactly 0. # If you didn't use epsilon-tolerance, fast_determinant might crash or return a tiny non-zero float.# print("Det:", m.fast_determinant())

---## Phase 3 — Olympiad Extension### Challenge: Matrix InversionImplement `inverse(self)`.1. Augment `self` with the Identity matrix.2. Run RREF.3. If the left side is NOT the identity, raise an Error ("Matrix is singular/non-invertible").4. Slice and return the right side as a new `Matrix`.

In [ ]:
# YOUR CODE HEREpass

### Chalkboard DefenseExplain why calculating the determinant via RREF takes $O(N^3)$ time, whereas recursive cofactor expansion takes $O(N!)$ time. What are the two nested loops occurring inside the `while` loop of RREF?<details><summary>💡 Sketch after a retrieval attempt</summary>The while loop runs $N$ times. Inside, we find a pivot ($O(N)$), and then we eliminate all other rows by subtracting a multiple of the pivot row. Subtracting a row from another row requires iterating over all $N$ columns. Doing this for $N$ rows takes $N \times N = O(N^2)$ time inside the while loop. Total time: $N \times O(N^2) = O(N^3)$.</details>---📚 **Next:** Algorithms 16 (K-Means Clustering)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_15_reduced_row_echelon_form', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='16_k_means_clustering.ipynb')